In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd
from collections.abc import Callable
from typing import Tuple

In [ ]:
def prepare_function(n, m) -> Callable:
    return lambda x: np.pow(x, n) - np.pow((1 - x), m)

def prepare_derivative(n, m) -> Callable:
    return lambda x: n * np.pow(x, n-1) + m * np.pow((1-x), m-1)

function = prepare_function(n=13, m=15)
derivative = prepare_derivative(n=13, m=15)

interval = (np.float64(0), np.float64(2))
y_lim = (-2, 2)
x_lim = (-1, 2)

xs = np.linspace(interval[0], interval[1], 1000)
ys = [function(x) for x in xs]
plt.ylim(y_lim)
plt.xlim(x_lim)
plt.xlabel('x')
plt.ylabel('y')
plt.title("Wizualizacja wykresu funkcji bazowej")
plt.plot(xs, ys, label="Funkcja bazowa")
plt.legend()
plt.savefig("base_function")
plt.close()

Metoda Siecznych

In [ ]:
def secant_method(function: Callable, start_point: np.float64, end_point: np.float64, stop: str, rho: np.float64, max_iterations:int = 1000) -> Tuple[np.float64, int]:
  if stop != 'residual'and stop != 'incremental':
    raise Exception("Wrong stop condition")
  x1 = min(start_point, end_point)
  x2 = max(start_point, end_point)
  
  for i in range(max_iterations):
    x1, x2 = x2, x2 - function(x2) * (x2-x1)/(function(x2)-function(x1))
    if stop == 'incremental' and abs(x2-x1) < rho:
      return x2, i+1 
    if stop == 'residual' and abs(function(x2)) < rho:
      return x2, i+1
    
  print(f"Secant method did not converge within {max_iterations} iterations")
  return x2, i+1

Metoda Newtona

In [ ]:
def newton_method(function: Callable, derivative: Callable, start_point:np.float64, stop: str, rho: np.float64, max_iterations:int = 1000) -> Tuple[np.float64, int]:
  if (stop != 'residual'and stop != 'incremental'):
      raise Exception("Wrong stop condition")
  
  x = start_point
  
  for i in range(max_iterations):
    dfx = derivative(x)
    x1 = x - function(x)/dfx
    if stop == 'incremental' and abs(x1-x) < rho:
      return x1, i+1
    if stop == 'residual' and abs(function(x1)) < rho:
      return x1, i+1
    
    x = x1
  
  print(f"Newton method did not converge within {max_iterations} iterations")
  return x1, i+1

Benchmark metod

In [ ]:
RHO = [1e-2, 1e-3, 1e-4, 1e-5, 1e-7, 1e-10, 1e-15]
START_POINTS = np.linspace(0.0, 2.0, 21)
STOP_CONDITIONS = ["incremental", "residual"]
INTERVAL_A = 0
INTERVAL_B = 2
TRUE_ROOT = 0.47522114835638004475
MAX_ITERATIONS = 10_000

results = {
    f"{method}_{stop}_{metric}": []
    for method in ("newton", "secant")
    for stop in STOP_CONDITIONS
    for metric in ("iterations", "error", "values")
}

output_dir = "results"
os.makedirs(output_dir, exist_ok=True)

for stop_cond in STOP_CONDITIONS:
    for rho in RHO:
        for start_point in START_POINTS:

            root_n = np.nan
            iters_n = np.nan
            error_n = np.nan
            try:
                root_n, iters_n = newton_method(
                    function, derivative, start_point,
                    stop_cond, np.float64(rho), MAX_ITERATIONS
                )
                if not np.isnan(root_n):
                    error_n = abs(root_n - TRUE_ROOT)
                else:
                    iters_n = np.nan
            except (ValueError, RuntimeError, OverflowError):
                pass

            results[f"newton_{stop_cond}_iterations"].append(
                {"start_point": start_point, "rho": rho, "iterations": iters_n}
            )
            results[f"newton_{stop_cond}_error"].append(
                {"start_point": start_point, "rho": rho, "error": error_n}
            )
            results[f"newton_{stop_cond}_values"].append(
                {"start_point": start_point, "rho": rho, "value": root_n}
            )

            for end_point in [INTERVAL_A, INTERVAL_B]:
                root_s = np.nan
                iters_s = np.nan
                error_s = np.nan

                if abs(start_point - end_point) < np.finfo(float).eps:
                    pass
                else:
                    try:
                        root_s, iters_s = secant_method(
                            function, np.float64(start_point),
                            np.float64(end_point), stop_cond,
                            np.float64(rho), MAX_ITERATIONS
                        )
                        if not np.isnan(root_s):
                            error_s = abs(root_s - TRUE_ROOT)
                        else:
                            iters_s = np.nan
                    except (ValueError, RuntimeError, OverflowError):
                        pass

                results[f"secant_{stop_cond}_iterations"].append(
                    {"start_point1": start_point, "start_point2": end_point,
                     "rho": rho, "iterations": iters_s}
                )
                results[f"secant_{stop_cond}_error"].append(
                    {"start_point1": start_point, "start_point2": end_point,
                     "rho": rho, "error": error_s}
                )
                results[f"secant_{stop_cond}_values"].append(
                    {"start_point1": start_point, "start_point2": end_point,
                     "rho": rho, "value": root_s}
                )

for key, data_list in results.items():
    if not data_list:
        continue
    df = pd.DataFrame(data_list)
    for col in df.columns:
        if col in {"value", "start_point", "start_point1", "start_point2"}:
            df[col] = df[col].round(8)
        elif col == "iterations":
            df[col] = df[col].astype("Int64")
    df.to_csv(os.path.join(output_dir, f"{key}.csv"), index=False)

print("Results saved to:", output_dir)


formatted_output_dir = "formatted_output"
os.makedirs(formatted_output_dir, exist_ok=True)

rho_labels = [f"10^-{int(-np.log10(r))}" if r < 1 else "10^0" for r in RHO]


def save_matrix_table(method_name, stop_condition, is_secant=False):
    key = f"{method_name}_{stop_condition}_values"
    value_entries = results[key]

    if is_secant:
        header = ["Punkty x1,x2"] + rho_labels
        point_pairs = sorted(
            {(round(v["start_point1"], 1), round(v["start_point2"], 1))
             for v in value_entries},
            key=lambda x: x[1] * 10 + x[0]
        )
        rows = []
        for x1, x2 in point_pairs:
            label = f"{x1},{x2}"
            row = [label]
            for rho in RHO:
                value = next(
                    (v["value"] for v in value_entries
                     if np.isclose(v["start_point1"], x1)
                     and np.isclose(v["start_point2"], x2)
                     and np.isclose(v["rho"], rho)),
                    np.nan
                )
                row.append(round(float(value), 8) if not np.isnan(value) else "")
            rows.append(row)
    else:
        header = ["Punkt x0"] + rho_labels
        rows = []
        for start_point in START_POINTS:
            row = [round(start_point, 1)]
            for rho in RHO:
                value = next(
                    (v["value"] for v in value_entries
                     if np.isclose(v["start_point"], start_point)
                     and np.isclose(v["rho"], rho)),
                    np.nan
                )
                row.append(round(float(value), 8) if not np.isnan(value) else "")
            rows.append(row)

    df_matrix = pd.DataFrame(rows, columns=header)
    filepath = os.path.join(formatted_output_dir,
                            f"{method_name}_{stop_condition}_matrix.csv")
    df_matrix.to_csv(filepath, index=False)
    print(f"Matrix saved: {filepath}")


for method in ["newton", "secant"]:
    for stop_cond in STOP_CONDITIONS:
        save_matrix_table(method, stop_cond, is_secant=(method == "secant"))

print("Matrix tables saved to:", formatted_output_dir)


heatmap_dir = "heatmaps"
os.makedirs(heatmap_dir, exist_ok=True)

metrics = ["error", "iterations"]
metrics_names = ["Błąd bezwzględny", "Liczba iteracji"]
methods = ["newton", "secant"]
methods_names = ["Metoda Newtona", "Metoda Siecznych"]
stop_conditions_names_map = {"incremental": "Przyrostowy", "residual": "Rezydualny"}


def plot_heatmap(df, x_col, y_col, x_col_name, y_col_name, value_col,
                 value_col_name, title, filename):
    pivot = df.pivot_table(index=y_col, columns=x_col, values=value_col)
    pivot = pivot[sorted(pivot.columns, reverse=True)]

    if isinstance(pivot.index[0], tuple):
        pivot = pivot.iloc[
            sorted(range(len(pivot.index)),
                   key=lambda i: (pivot.index[i][0], pivot.index[i][1]))
        ]
    else:
        pivot = pivot.sort_index()

    data = pivot.values.astype(float)
    row_labels = [str(r) for r in pivot.index]
    col_labels = [str(c) for c in pivot.columns]

    n_rows, n_cols = data.shape
    fig, ax = plt.subplots(figsize=(12, max(6, n_rows * 0.4)))

    cmap = plt.get_cmap("plasma")
    vmin, vmax = np.nanmin(data), np.nanmax(data)
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    ax.imshow(data, cmap=cmap, norm=norm, aspect="auto")
    cbar = plt.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax)
    cbar.set_label(value_col_name)

    ax.set_xticks(range(n_cols))
    ax.set_xticklabels(col_labels, rotation=45, ha="right")
    ax.set_yticks(range(n_rows))
    ax.set_yticklabels(row_labels)

    for i in range(n_rows):
        for j in range(n_cols):
            val = data[i, j]
            if np.isnan(val):
                text = ""
            elif abs(val) >= 1e4 or (abs(val) < 1e-3 and val != 0):
                text = f"{val:.2e}"
            elif val == int(val):
                text = str(int(val))
            else:
                text = f"{val:.4g}"

            bg_rgba = cmap(norm(val))
            luminance = 0.299 * bg_rgba[0] + 0.587 * bg_rgba[1] + 0.114 * bg_rgba[2]
            ax.text(j, i, text, ha="center", va="center",
                    fontsize=7, color="white" if luminance < 0.5 else "black")

    ax.set_title(title, pad=12)
    ax.set_xlabel(x_col_name)
    ax.set_ylabel(y_col_name)
    ax.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=0.5)
    ax.tick_params(which="minor", bottom=False, left=False)

    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()
    print(f"Saved: {filename}")


for method_i, method in enumerate(methods):
    for stop in STOP_CONDITIONS:
        stop_name = stop_conditions_names_map[stop]
        for m_i, metric in enumerate(metrics):
            csv_name = f"{method}_{stop}_{metric}.csv"
            csv_path = os.path.join(output_dir, csv_name)

            if not os.path.exists(csv_path):
                print(f"Missing: {csv_name}")
                continue

            df = pd.read_csv(csv_path)

            if method == "newton":
                plot_heatmap(
                    df,
                    x_col="rho", y_col="start_point",
                    x_col_name="Współczynnik precyzji ρ",
                    y_col_name="Punkt początkowy x0",
                    value_col=metric,
                    value_col_name=metrics_names[m_i],
                    title=f"{methods_names[method_i]} ({stop_name}) - {metrics_names[m_i]}",
                    filename=os.path.join(heatmap_dir,
                                          f"{method}_{stop}_{metric}_heatmap.png"),
                )
            else:
                df["start_pair"] = df.apply(
                    lambda row: f"({row['start_point1']:.1f}, {row['start_point2']:.1f})", axis=1
                )
                df["_sort_key"] = df["start_point2"] * 1000 + df["start_point1"]  # numeric sort key
                pair_order = (
                    df[["start_pair", "_sort_key"]]
                    .drop_duplicates()
                    .sort_values("_sort_key")["start_pair"]
                    .tolist()
                )
                df["start_pair"] = pd.Categorical(df["start_pair"], categories=pair_order, ordered=True)
                plot_heatmap(
                    df,
                    x_col="rho", y_col="start_pair",
                    x_col_name="Współczynnik precyzji ρ",
                    y_col_name="Punkty początkowe x1, x2",
                    value_col=metric,
                    value_col_name=metrics_names[m_i],
                    title=f"{methods_names[method_i]} ({stop_name}) - {metrics_names[m_i]}",
                    filename=os.path.join(heatmap_dir,
                                          f"{method}_fixed_{stop}_{metric}_heatmap.png"),
                )

print("Heatmaps saved to:", heatmap_dir)